# Rafael TCC - Projeto 04

Análise de consitência dos dados de touros Nelore presentes na plataforma CRV, segundo o programa de melhoramento

## Bibliotecas

In [1]:
import os
import pandas as pd

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


## Extração, leitura e transformação dos dados

### Leitura dos dados

In [2]:
# Leitura do arquivo de dados
programa = 'ANCP'
df = pd.read_csv(f'Dados/CSV/NeloreCRV_{programa}.csv')

### Verificação da existência de observações duplicadas

In [3]:
# Verificando se a observações duplicadas
duplicados = df.duplicated().sum()
print(f"Linhas duplicadas no dataset: {duplicados}")

# Se houver uma chave única (ex: 'crv_code' ou 'rgd'):
print(f"Códigos duplicados: {df['crv_code'].duplicated().sum()}")

Linhas duplicadas no dataset: 0
Códigos duplicados: 0


### Verificação preliminar dos dados

In [4]:
# Verificação preliminar dos dados (nomes das variáveis, tipo de dado e a existência de valores nulos)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 63 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   crv_code            120 non-null    float64
 1   nome_oficial        120 non-null    str    
 2   raca                120 non-null    str    
 3   rgd                 120 non-null    str    
 4   data_nascimento     120 non-null    str    
 5   preco_convencional  119 non-null    float64
 6   selos               120 non-null    str    
 7   ANCP_MGTe_DEP       120 non-null    float64
 8   ANCP_MGTe_TOP       120 non-null    float64
 9   ANCP_D3P_ACC        120 non-null    float64
 10  ANCP_D3P_DEP        120 non-null    float64
 11  ANCP_D3P_TOP        120 non-null    float64
 12  ANCP_DACAB_ACC      120 non-null    float64
 13  ANCP_DACAB_DEP      120 non-null    float64
 14  ANCP_DACAB_TOP      120 non-null    float64
 15  ANCP_DAOL_ACC       120 non-null    float64
 16  ANCP_DAOL_DEP      

### Renomeação das variáveis (colunas)

In [5]:
# Simplificacão e alteração dos nomes das variáveis de registro e semem para caixa alta 
novos_nomes = {
    'crv_code': 'CRV_COD',
    'nome_oficial': 'NOME',
    'raca': 'RACA',
    'rgd': 'RGD',
    'data_nascimento': 'DNASC',
    'preco_convencional': 'SEMEM',
    'selos': 'SELOS'
}

df.rename(columns=novos_nomes, inplace=True)

In [6]:
# Removendo o prefixo ANCP e ANCP_D das variáveis relacionadas a avalição genética dos animais
df.columns = [c.replace('ANCP_D', '') for c in df.columns]
df.columns = [c.replace('ANCP_', '') for c in df.columns]

In [7]:
# Verificação preliminar das dimensões dos dados
print(f"Dimensões do dataset: {df.shape[0]} touros x {df.shape[1]} variáveis")

Dimensões do dataset: 120 touros x 63 variáveis


In [8]:
# Eliminação das observações cuja variável SEMEM apresentaram valor nulo
df_notnull = df[df['SEMEM'].notna()]

print(f"Dimensões do dataset: {df_notnull.shape[0]} touros x {df_notnull.shape[1]} variáveis")

Dimensões do dataset: 119 touros x 63 variáveis


In [9]:
# Verificação dos dados após as alterações nos nomes das variáveis e eliminação dos valores nulos na variável SEMEM
df_notnull.info()

<class 'pandas.DataFrame'>
RangeIndex: 119 entries, 0 to 118
Data columns (total 63 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   CRV_COD      119 non-null    float64
 1   NOME         119 non-null    str    
 2   RACA         119 non-null    str    
 3   RGD          119 non-null    str    
 4   DNASC        119 non-null    str    
 5   SEMEM        119 non-null    float64
 6   SELOS        119 non-null    str    
 7   MGTe_DEP     119 non-null    float64
 8   MGTe_TOP     119 non-null    float64
 9   3P_ACC       119 non-null    float64
 10  3P_DEP       119 non-null    float64
 11  3P_TOP       119 non-null    float64
 12  ACAB_ACC     119 non-null    float64
 13  ACAB_DEP     119 non-null    float64
 14  ACAB_TOP     119 non-null    float64
 15  AOL_ACC      119 non-null    float64
 16  AOL_DEP      119 non-null    float64
 17  AOL_TOP      119 non-null    float64
 18  CAR_ACC      119 non-null    float64
 19  CAR_DEP      119 no

### Avaliando a distribuição de valores da variável SEMEM

In [10]:

df_cols = df_notnull.columns[df.columns.isin(['SEMEM'])]

# Passa a lista inteira de colunas, aplica o describe com os percentis e transpõe (.T)
df_resumo_SEMEM = (
    df[df_cols]
    .describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.90, 0.95])
    .T
    .round(2)
)

display(df_resumo_SEMEM)

,count,mean,std,min,5%,25%,50%,75%,90%,95%,max
SEMEM,119.0,38.87,43.98,0.0,21.58,31.2,31.2,34.4,38.9,48.7,346.5


In [11]:
# Eliminação dos valores de doses de semem não válidos (<= 0 e >= 100)
df_valid = df_notnull[(df_notnull['SEMEM'] > 0) & (df_notnull['SEMEM'] <= 100 )].copy()

# Informações dos dados após processo de consistência
print(f"Touros originais com preço: {len(df_notnull)}")
print(f"Touros após remoção de outliers: {len(df_valid)} (removidas {len(df_notnull) - len(df_valid)} observações)")
print(f"Touros com preço convencional válido: {len(df_valid)} ({len(df_valid)/len(df)*100:.1f}% do catálogo)")

Touros originais com preço: 119
Touros após remoção de outliers: 115 (removidas 4 observações)
Touros com preço convencional válido: 115 (95.8% do catálogo)


In [12]:
df_cols = df_notnull.columns[df.columns.isin(['SEMEM'])]

# Passa a lista inteira de colunas, aplica o describe com os percentis e transpõe (.T)
df_resumo_SEMEM = (
    df[df_cols]
    .describe(percentiles=[0.05, 0.25, 0.50, 0.75, 0.90, 0.95])
    .T
    .round(2)
)

display(df_resumo_SEMEM)

,count,mean,std,min,5%,25%,50%,75%,90%,95%,max
SEMEM,119.0,38.87,43.98,0.0,21.58,31.2,31.2,34.4,38.9,48.7,346.5


### Exportação dos dados válidos para o arquivo NeloreCRV_ancp_validos.csv

In [13]:
df_valid.to_csv(f'Dados/CSV/NeloreCRV_{programa}_valido.csv', index=False)

print(f'Arquivo salvo com sucesso!!')

Arquivo salvo com sucesso!!
